# Sysmon CNN-LSTM：勒索软件 / 正常软件二分类

适配 **Preprocess v2.0.1**，默认 `[batch,64,148]`，system 视图。主指标按整次采集计算。
这是离线主机行为分类，不是论文 iCNN-LSTM+ 的严格复现，也不包含增量学习。

**开始前**：将完整前处理文件夹上传 Drive → `MyDrive/SysmonResearch/avos-blackbasta-benign-v1/`；选择 GPU 运行时，按顺序执行。
只需上传这个 Notebook，辅助代码已内嵌。不要上传恶意样本 EXE。

当前数据核对：53 次候选采集、1326 个 system 片段。正常软件不同版本/安装/卸载是否同组仍需人工复核。
默认结果属于 hash/group 留出初步实验；不能宣称泛化到从未见过的正常软件。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 配置（主要修改这里）

`DATA_DIR` 指向前处理目录。也可以设置 `DATA_ZIP` 为 Drive 上的数据 ZIP；代码会解压并自动定位目录。
软件分组修正须在前处理 annotation 流程完成，重新生成数据；开关只记录是否复核，不会自动分组。
正式对比模型须复用 `SPLITS_PATH`，不要根据测试成绩换随机种子。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid

PROJECT = Path('/content/drive/MyDrive/SysmonResearch')
DATA_DIR = PROJECT / 'avos-blackbasta-benign-v1'
DATA_ZIP = None  # 例如 PROJECT / 'avos-blackbasta-benign-v1.zip'
SPLITS_PATH = PROJECT / 'splits_family_seed20261001.csv'
SEED = 20261001
VIEW = 'system'  # related 依赖已知样本进程，限辅助实验；切换时使用新的输出目录
EPOCHS = 40
BATCH_SIZE = 32
SOFTWARE_GROUPS_REVIEWED = False
OUTPUT = PROJECT / 'experiments' / ('cnn_lstm_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
print('数据:', DATA_DIR, '\n结果:', OUTPUT)


## 内嵌代码与依赖

下一单元将随 Notebook 打包的开源 Python 源码展开到临时目录，可以打开目录逐文件审查。它不下载远程代码。
优先使用 Colab 预装依赖；缺失时才安装，并重启运行时。实际版本会保存到实验记录。


In [ ]:
import base64, zlib, json, sys, tempfile, importlib.util
CODE_DIR = Path(tempfile.mkdtemp(prefix='sysmon_cnn_lstm_'))
EMBEDDED_SOURCE = '...'
source_files = json.loads(zlib.decompress(base64.b64decode(EMBEDDED_SOURCE)))
for name, source in source_files.items():
    destination = CODE_DIR / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE_DIR))
missing = [n for n in ['tensorflow', 'numpy', 'sklearn', 'matplotlib'] if importlib.util.find_spec(n) is None]
if missing:
    raise RuntimeError(f'缺少依赖 {missing}；安装 tensorflow scikit-learn matplotlib 后重启运行时，再从头执行。')
import tensorflow as tf
import training as T
print('TensorFlow:', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))
print('展开源码:', CODE_DIR)
if not tf.config.list_physical_devices('GPU'):
    print('当前为 CPU：可以验证流程，正式训练可切换 GPU。')


In [ ]:
# 可选：从 ZIP 解压到 Colab 临时磁盘。原数据保持不变。
if DATA_ZIP is not None:
    import zipfile, shutil
    staging = Path(tempfile.mkdtemp(prefix='sysmon_data_'))
    with zipfile.ZipFile(DATA_ZIP) as archive:
        if sum(i.file_size for i in archive.infolist()) > 8 * 1024**3:
            raise ValueError('解压后超过 8 GiB，请先检查压缩包和内存容量')
        for info in archive.infolist():
            target = (staging / info.filename).resolve()
            if not target.is_relative_to(staging.resolve()) or '\\' in info.filename:
                raise ValueError('非法 ZIP 路径')
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
    candidates = list(staging.rglob('build.json'))
    if len(candidates) != 1:
        raise ValueError('ZIP 应只包含一份完整的前处理数据')
    DATA_DIR = candidates[0].parent
print('实际数据目录:', DATA_DIR)


## 读取和校验数据，固定分组划分

只使用候选序列。SHA256、家族、软件名等仅作标签/审计，不作模型特征。
默认正常类作为一层、恶意类按家族分层，保持 group_id 不跨集合；每个家族至少 3 组。
固定 schema 已处理未知类别，不重新拟合词表、不使用标签均值编码、也不做 SMOTE。
读取器检查校验和、序列顺序、窗口边界、右侧补齐和分组一致性；大数据时可能需要数分钟及较多内存。


In [ ]:
data = T.load_data(DATA_DIR, view=VIEW)
indices, split_summary = T.prepare_splits(data, SPLITS_PATH, SEED)
print('X:', data['x'].shape, 'mask:', data['mask'].shape)
print(json.dumps(split_summary, ensure_ascii=False, indent=2))
if not SOFTWARE_GROUPS_REVIEWED:
    print('解释边界：软件产品/安装卸载分组尚未确认，本次为初步 hash/group 留出实验。')


## 训练：多尺度 CNN → LSTM → 二分类

每个片段的顺序保留，训练时只打乱片段的出现顺序。显式 mask 防止 PAD 影响卷积/LSTM。
训练集按运行长度和类别加权；验证集按运行长度加权。早停/最佳模型只看验证损失。
运行概率固定为片段概率的算术平均；阈值仅在验证运行上最大化 F2，同时记录 FPR。


In [ ]:
model, decision = T.train(data, indices, OUTPUT, epochs=EPOCHS, batch_size=BATCH_SIZE,
                          seed=SEED, software_groups_reviewed=SOFTWARE_GROUPS_REVIEWED)
import shutil
shutil.copy2(SPLITS_PATH, OUTPUT / 'splits.csv')
T.save_json(OUTPUT / 'split_summary.json', split_summary)
print(json.dumps(decision, ensure_ascii=False, indent=2))


## 最终测试（独立单元）

模型架构、checkpoint、聚合方式和阈值已冻结后再运行。不要用测试集挑选 epoch、阈值或种子。
主指标为运行级；片段继承运行弱标签，因此片段结果不表示逐事件恶意性判断。
当前样本数很小，必须同时报告 n、混淆矩阵、Recall、Precision、FPR/FNR，不能只报告高准确率。


In [ ]:
report = T.evaluate_test(model, data, indices, OUTPUT, decision)
print(json.dumps(report, ensure_ascii=False, indent=2))
fig = T.plot_results(OUTPUT)
import matplotlib.pyplot as plt
plt.show()


## 保存模型重载检查及结果下载

重载前必须 import training 注册 ApplyMask。下面验证保存前后预测一致，不再调整模型。
模型输出是实验分类分数，并未单独做概率校准。将来未知标签日志的在线推理需要另建入口。


In [ ]:
import numpy as np
restored = tf.keras.models.load_model(OUTPUT / 'best.keras')
example_indices = indices['validation'][:8]
before = T.predict(model, data, example_indices)
after = T.predict(restored, data, example_indices)
np.testing.assert_allclose(before, after, atol=1e-6, rtol=1e-6)
print('模型保存/重载一致。结果目录:', OUTPUT)
archive = shutil.make_archive('/content/' + OUTPUT.name, 'zip', root_dir=OUTPUT)
from google.colab import files
files.download(archive)


## 研究解释与下一步

- 当前 LSTM 学习最多 64 个事件的片段内模式，不自动跨片段记住完整 300 秒；需要 CNN-only / LSTM-only 等对照验证其收益。
- 继承 Ispahany（2025）和 Akhtar（2022）的 CNN/LSTM 研究方向，但输入、网络、离线采集和评估单位均有变化；不可当作严格复现后比较分数。
- 同产品正常软件、样本近重复、环境差异和弱标签仍需审查；相关视图依赖已知根进程，不等同于部署时未知来源的 system 视图。
- 未实现增量学习、家族多分类、实时部署或论文性能保证。

参考：[Ispahany 2025](https://doi.org/10.1109/ACCESS.2025.3569635)、[Akhtar 2022](https://doi.org/10.3390/sym14112308)、[TensorFlow masking](https://www.tensorflow.org/guide/keras/understanding_masking_and_padding)、[TESSERACT](https://www.usenix.org/conference/usenixsecurity19/presentation/pendlebury)。详细说明见随附 README。
